In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
# Delete the past output
import shutil
from pathlib import Path

out = Path("outputs/smol-lora")
if out.exists():
  shutil.rmtree(out)

In [ ]:
import os, shutil
from kaggle_secrets import UserSecretsClient

# Secrets
secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")

# Pull latest code
repo_path = "/kaggle/working/mini-artificer"
if os.path.exists(repo_path):
    %cd {repo_path}
    !git pull origin main
else:
    !git clone https://github.com/reuel-ly/mini-artificer.git {repo_path}
    %cd {repo_path}

# Install dependencies
!pip install -q transformers peft trl datasets bitsandbytes accelerate wandb

print("Ready.")

In [ ]:
import torch
n_gpu = max(1, torch.cuda.device_count())
print(f"Launching on {n_gpu} GPU(s)")
!accelerate launch --num_processes {n_gpu} --mixed_precision fp16 train.py

In [ ]:
import os
print(os.path.exists("/kaggle/working/mini-artificer/outputs/smol-lora"))

In [ ]:
 # 1. List output dir
import os
print(os.listdir("/kaggle/working/mini-artificer/outputs/smol-lora"))
# 2. Check chat template was saved
import json
with open("/kaggle/working/mini-artificer/outputs/smol-lora/tokenizer_config.json") as f:
  cfg = json.load(f)
print("Has generation markers:", "{% generation %}" in cfg.get("chat_template", ""))
# 3. Raw input debug (add to inference.py or run inline)
from inference import _load_model_and_tokenizer, WEATHER_TOOL_SCHEMA
from transformers import AutoTokenizer
output_dir = "/kaggle/working/mini-artificer/outputs/smol-lora"
_, tokenizer = _load_model_and_tokenizer(output_dir)
messages = [
  {"role": "system", "content": "You are a helpful assistant with access to the following functions. Use them if required -\n" + str(WEATHER_TOOL_SCHEMA)},
  {"role": "user", "content": "What is the weather in Manila?"},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt")
print("=== RAW INPUT ===")
print(tokenizer.decode(inputs["input_ids"][0]))
print("=================")

In [ ]:
# Check what's actually saved in your output directory
import os
print(os.listdir("/kaggle/working/mini-artificer/outputs/smol-lora"))

# Load and print the saved chat template
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(
    "/kaggle/working/mini-artificer/outputs/smol-lora"
)
print(tok.chat_template)

In [ ]:
from transformers import AutoTokenizer
from chat_template import patch_chat_template

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")
tokenizer = patch_chat_template(tokenizer)

sample_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "What is the weather in Manila?"},
    {"role": "assistant", "content": '<functioncall> {"name": "get_weather", "arguments": {"location": "Manila"}}'}
]

formatted = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
)
print(formatted)

In [ ]:
print("eos:", tokenizer.eos_token_id, tokenizer.eos_token)
print("im_end:", tokenizer.convert_tokens_to_ids(""))
print("eot:", tokenizer.convert_tokens_to_ids("<|endoftext|>"))